In [ ]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [ ]:
model_path = "face_landmarker.task"


In [ ]:
import cv2
import mediapipe as mp 

In [ ]:
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions

In [ ]:
# try blendshapes 
# https://ai.google.dev/edge/mediapipe/solutions/vision/face_landmarker/python
options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    num_faces=1,
    output_face_blendshapes=True
)

landmarker = FaceLandmarker.create_from_options(options)

In [ ]:
cap = cv2.VideoCapture(0)

In [ ]:
from pythonosc.udp_client import SimpleUDPClient

while True:
    ret, frame = cap.read()
    if not ret:
        break

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=frame
    )

    result = landmarker.detect(mp_image)

    if result.face_blendshapes:
        blendshapes = result.face_blendshapes[0]

        # Convert to dictionary (IMPORTANT)
        data = {b.category_name: b.score for b in blendshapes}

        # Print a few useful ones
        print(
            "brow:", data.get("browInnerUp", 0),
            "smile:", data.get("mouthSmileLeft", 0),
            "jaw:", data.get("jawOpen", 0)
        )
        
        client.send_message("/face/brow", data.get("browInnerUp", 0))
        client.send_message("/face/jaw", data.get("jawOpen", 0))
        client.send_message("/face/smile", data.get("mouthSmileLeft", 0))

    cv2.imshow("cam", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
#pip install python-osc

In [ ]:
import cv2
import mediapipe as mp
from pythonosc.udp_client import SimpleUDPClient

#SETUP

client = SimpleUDPClient("127.0.0.1", 8000)

# MediaPipe setup
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='face_landmarker.task'),
    output_face_blendshapes=True,
    num_faces=1
)

landmarker = FaceLandmarker.create_from_options(options)

cap = cv2.VideoCapture(1)

# MAIN LOOP
import pynput
from pynput import keyboard
listener = keyboard.Listener(on_press=on_press)
listener.start()
def on_press(key):
    if key == keyboard.Key.esc:
        return True
    
if on_press:
    ret, frame = cap.read()
    # if not ret:
    #     break

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=frame
    )

    result = landmarker.detect(mp_image)

    if result.face_blendshapes:
        blendshapes = result.face_blendshapes[0]

        data = {b.category_name: b.score for b in blendshapes}

        # brow = data.get("browInnerUp", 0)
        # jaw = data.get("jawOpen", 0)
        # smile = data.get("mouthSmileLeft", 0)

        # print("brow:", brow, "jaw:", jaw, "smile:", smile)
        # print(data)

        #----------------------------
        # SET STATES
        #-----------------------------
        valence = data["mouthSmileLeft"] + data["mouthSmileRight"] - data["mouthFrownLeft"] - data["mouthFrownRight"]
        thinking = data["eyeLookUpLeft"] + data["eyeLookUpRight"]
        arousel = data["jawOpen"] + data["eyeWideLeft"] + data["eyeWideRight"] + data["browInnerUp"]
        anxious = data["eyeSquintLeft"] + data["eyeSquintRight"] + data["browDownLeft"] + data["browDownRight"] + data["mouthPressLeft"] + data["mouthPressRight"]
    
        #----------------------------
        # TEST CLICK AND GET RESULT 
        #-----------------------------



        #-----------------------------
        # send OSC messages
        #-----------------------------
        # client.send_message("/face/brow", brow)
        # client.send_message("/face/jaw", jaw)
        # client.send_message("/face/smile", smile)
        # landmarks = result.face_landmarks[0]
        # client.send_message("/pos/brow_x", landmarks[65].x)
        # client.send_message("/pos/brow_y", landmarks[65].y)

    # cv2.imshow("cam", frame)

    if cv2.waitKey(1) & 0xFF == 27:

        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=frame
    )

    result = landmarker.detect(mp_image)

    if result.face_blendshapes:
        blendshapes = result.face_blendshapes[0]

        data = {b.category_name: b.score for b in blendshapes}


In [ ]:
# import pynput
# from pynput import keyboard

# capture_now = False

# def on_press(key):
#     global capture_now

#     if key == keyboard.Key.space:   # press SPACE to capture
#         capture_now = True
#         print("📸 Capture triggered")

#     if key == keyboard.Key.esc:
#         return False  # stop listener

# listener = keyboard.Listener(on_press=on_press)
# listener.start()

In [20]:
import cv2
import mediapipe as mp
from pythonosc.udp_client import SimpleUDPClient
import pynput
from pynput import keyboard

#SETUP

client = SimpleUDPClient("127.0.0.1", 8000)

# MediaPipe setup
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='face_landmarker.task'),
    output_face_blendshapes=True,
    num_faces=1
)

landmarker = FaceLandmarker.create_from_options(options)

cap = cv2.VideoCapture(1)

In [23]:
capture_now = False

def on_press(key):
    global capture_now

    if key == keyboard.Key.space:   # press SPACE to capture
        capture_now = True
        print("📸 Capture triggered")

    if key == keyboard.Key.esc:
        return False  # stop listener

listener = keyboard.Listener(on_press=on_press)
listener.start()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if capture_now:
        capture_now = False  # reset flag

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=frame
        )

        result = landmarker.detect(mp_image)

        if result.face_blendshapes:
            blendshapes = result.face_blendshapes[0]
            data = {b.category_name: b.score for b in blendshapes}

            # ----------------------------
            # STATES
            # ----------------------------
            valence = data["mouthSmileLeft"] + data["mouthSmileRight"] - data["mouthFrownLeft"] - data["mouthFrownRight"]
            thinking = data["eyeLookUpLeft"] + data["eyeLookUpRight"]
            arousal = data["jawOpen"] + data["eyeWideLeft"] + data["eyeWideRight"] + data["browInnerUp"]
            anxious = data["eyeSquintLeft"] + data["eyeSquintRight"] + data["browDownLeft"] + data["browDownRight"] + data["mouthPressLeft"] + data["mouthPressRight"]
            stability = 0.5

            # convert to percentage 
            valence_percent = max(0, min(100, valence * 100))
            thinking_percent = max(0, min(100, thinking * 100)) 
            arousal_percent = max(0, min(100, arousal * 100))
            anxious_percent = max(0, min(100, anxious * 100))
            stability_percent = max(0, min(100, stability * 100))

            

            # get suboptimal results 
            score = 0.25*valence_percent + 0.25*arousal_percent + 0.2*thinking_percent + 0.2*anxious_percent + 0.1*stability_percent
            score_percent = max(0, min(100, score * 100))  # convert to percentage

            # print results 
            print("------ STATS ------") 
            print("Valence:", round(valence_percent, 3))
            print("Thinking:", round(thinking_percent, 3))
            print("Arousal:", round(arousal_percent, 3))
            print("Anxious:", round(anxious_percent, 3))  
            print("------ RESULT ------")
            print("Overall Score:", round(score, 2), "%")

            # ----------------------------
            # FINAL SCORE
            # ----------------------------

            # face_score = 45
            voice_score = 60
            text_score = 65

            sum = 0.5*score + 0.2*voice_score + 0.3*text_score
            final_score = max(0, min(100, sum))

            # authenticity
            authenticity = 1 - final_score

            # Print results
            print("")
            print("----------FINAL STATS----------")
            print("Face Score:", score, "%")
            print("Voice Score:", voice_score, "%")     
            print("Text Score:", text_score, "%")
            print("------ RESULT ------")
            print("Overall Score:", round(final_score), "%")
            if final_score >=50: 
                print("Optimal")
                print("Authenticity: Low (", round(authenticity, 2), "%)")
            else:
                print("Suboptimal")

            
    cv2.imshow("cam", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

📸 Capture triggered
------ STATS ------
Valence: 0
Thinking: 2.488
Arousal: 90.022
Anxious: 26.18
------ RESULT ------
Overall Score: 33.24 %

----------FINAL STATS----------
Face Score: 33.23896946472814 %
Voice Score: 60 %
Text Score: 65 %
------ RESULT ------
Overall Score: 48 %
Suboptimal
📸 Capture triggered
------ STATS ------
Valence: 0
Thinking: 14.621
Arousal: 18.856
Anxious: 63.382
------ RESULT ------
Overall Score: 25.31 %

----------FINAL STATS----------
Face Score: 25.31456465017982 %
Voice Score: 60 %
Text Score: 65 %
------ RESULT ------
Overall Score: 44 %
Suboptimal
📸 Capture triggered
------ STATS ------
Valence: 46.968
Thinking: 3.209
Arousal: 74.14
Anxious: 80.756
------ RESULT ------
Overall Score: 52.07 %

----------FINAL STATS----------
Face Score: 52.06989264013828 %
Voice Score: 60 %
Text Score: 65 %
------ RESULT ------
Overall Score: 58 %
Optimal
Authenticity: Low ( -56.53 %)
📸 Capture triggered
------ STATS ------
Valence: 5.141
Thinking: 4.422
Arousal: 48.9